# 3. Natural Cluster Discovery for AI Artifacts

This notebook discovers natural groupings (clusters) of AI coding tool artifacts based on their semantic content.

**Pipeline Overview:**
```
Input: 768-dim embeddings from filtered_common_embeddings.npz
    ↓
UMAP Dimensionality Reduction (768 → N dims)
    ↓
HDBSCAN Clustering (density-based, automatic K)
    ↓
Output: Cluster assignments + fitted model for OSS predictions
```

**Input:** Pre-filtered data from `2. embedding_filtration.ipynb`  
**Output:** Cluster model in `data/cluster_model/` for `oss_cluster_assignment.ipynb`

## Setup

In [ ]:
# IMPORTANT: Set OpenMP environment variables BEFORE any other imports to prevent kernel crash on macOS ARM
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Suppress duplicate OpenMP library conflicts
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"  # Replacement for deprecated omp_set_nested
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import pickle
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

# Try importing HDBSCAN (better for variable density clusters)
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("HDBSCAN not installed. Install with: pip install hdbscan")

# Add src to path for imports
sys.path.insert(0, os.path.abspath('..'))

print(f"HDBSCAN available: {HAS_HDBSCAN}")

## Configuration

In [ ]:
# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / 'data'
CLUSTER_MODEL_DIR = DATA_DIR / 'cluster_model'

# Random seed for reproducibility
RANDOM_STATE = 42

# UMAP parameters for 2D visualization
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

print(f"Data directory: {DATA_DIR}")
print(f"Cluster model directory: {CLUSTER_MODEL_DIR}")

## Load Filtered Data

Load pre-filtered embeddings and metadata from notebook `2. embedding_filtration.ipynb`.

In [ ]:
# Load pre-filtered data from notebook 2

# Load embeddings (allow_pickle=True needed for string arrays like global_file_ids)
embeddings_path = DATA_DIR / 'filtered_common_embeddings.npz'
data = np.load(embeddings_path, allow_pickle=True)
embeddings = data['embeddings']
global_file_ids = data['global_file_ids']

# Load metadata
metadata_path = DATA_DIR / 'filtered_common_metadata.csv'
metadata = pd.read_csv(metadata_path)

print(f"Loaded {len(embeddings):,} embeddings from {embeddings_path.name}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Loaded {len(metadata):,} metadata rows from {metadata_path.name}")
print(f"\nMetadata columns: {list(metadata.columns)}")

## HDBSCAN Clustering Pipeline

> **Note:** UMAP dimensionality reduction is now part of the optimization process.
> We test multiple UMAP dimensionalities to find the optimal one.

## HDBSCAN Hyperparameter Optimization

### Why This Cell is Needed

**Problem:** HDBSCAN clustering quality depends heavily on:
1. **Dimensionality** - How many UMAP dimensions to reduce to (5D? 50D? 300D?)
2. **Hyperparameters** - `min_cluster_size`, `min_samples`, and `cluster_selection_method`

Choosing these manually is guesswork. This cell uses **Optuna** (Bayesian optimization) to systematically search for the best combination.

### What It Does

| Step | Action |
|------|--------|
| **1. Pre-compute UMAP** | Reduce embeddings to 20 different dimensionalities (5D → 700D) |
| **2. Optimize per dimension** | For each UMAP output, run 15 Optuna trials to find best HDBSCAN params |
| **3. Compare & select** | Find which (dimensionality + params) gives best clustering |

### Configuration Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `N_TRIALS_PER_DIM` | 15 | Optuna trials per dimensionality |
| `SUBSAMPLE_SIZE` | 3000 | Speed up optimization (full data used later) |
| `OPTIMIZE_FOR` | 'balanced' | Objective: silhouette × coverage (not just silhouette) |

### Why "Balanced" Optimization?

Pure silhouette optimization often produces few tight clusters with many noise points. The balanced objective rewards:
- **High silhouette** (well-separated clusters)
- **High coverage** (fewer points marked as noise)
- **Reasonable cluster count** (3-50 clusters)

### Output

Saves optimal parameters to variables for the next cell:
- `BEST_N_COMPONENTS`, `BEST_MIN_CLUSTER_SIZE`, `BEST_MIN_SAMPLES`, `BEST_METHOD`

In [ ]:
# HDBSCAN Hyperparameter Optimization using Optuna
# Approach: Separate optimization for each UMAP dimensionality, then compare

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Try to import optuna
try:
    import optuna
    HAS_OPTUNA = True
except ImportError:
    print("Installing optuna...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'optuna', '-q'])
    import optuna
    HAS_OPTUNA = True

optuna.logging.set_verbosity(optuna.logging.WARNING)

if HAS_HDBSCAN:
    print("=" * 70)
    print("OPTIMIZATION: Separate search per UMAP dimensionality")
    print("=" * 70)
    
    # ============================================================
    # CONFIGURATION
    # ============================================================
    N_TRIALS_PER_DIM = 15       # Trials per dimensionality
    SUBSAMPLE_SIZE = 3000       # Subsample for speed
    OPTIMIZE_FOR = 'balanced'   # 'silhouette', 'dbcv', or 'balanced'
    
    # UMAP dimensionality options (extended to test higher dims)
    N_COMPONENTS_OPTIONS = [5, 10, 20, 30, 50, 70, 90, 110, 130, 150, 170, 190, 210, 230, 250, 300, 400, 500, 600, 700]
    # ============================================================
    
    # Subsample
    if SUBSAMPLE_SIZE and len(embeddings) > SUBSAMPLE_SIZE:
        np.random.seed(42)
        subsample_idx = np.random.choice(len(embeddings), SUBSAMPLE_SIZE, replace=False)
        embeddings_subsample = embeddings[subsample_idx]
        print(f"\nSubsampling: {len(embeddings):,} → {SUBSAMPLE_SIZE:,} points")
    else:
        embeddings_subsample = embeddings
        print(f"\nUsing full dataset: {len(embeddings):,} points")
    
    # ============================================================
    # STEP 1: Pre-compute UMAP reductions
    # ============================================================
    print(f"\n" + "=" * 70)
    print("STEP 1: Pre-computing UMAP reductions")
    print("=" * 70)
    
    umap_cache = {}
    for n_comp in N_COMPONENTS_OPTIONS:
        print(f"  UMAP n_components={n_comp}...", end=" ", flush=True)
        reducer = umap.UMAP(
            n_neighbors=15,
            n_components=n_comp,
            min_dist=0.0,
            metric='cosine',
            random_state=RANDOM_STATE,
            n_jobs=1
        )
        umap_cache[n_comp] = reducer.fit_transform(embeddings_subsample)
        print(f"→ {umap_cache[n_comp].shape}")
    
    print(f"\n✓ Pre-computed {len(N_COMPONENTS_OPTIONS)} UMAP reductions")
    
    # ============================================================
    # STEP 2: Separate optimization for each dimensionality
    # ============================================================
    print(f"\n" + "=" * 70)
    print("STEP 2: Optimizing HDBSCAN for each dimensionality")
    print("=" * 70)
    print(f"\nTrials per dimension: {N_TRIALS_PER_DIM}")
    print(f"Total trials: {N_TRIALS_PER_DIM * len(N_COMPONENTS_OPTIONS)}")
    
    all_results = []  # Store all trials across all dimensions
    best_per_dim = {} # Store best result per dimension
    
    for n_comp in N_COMPONENTS_OPTIONS:
        print(f"\n{'─' * 70}")
        print(f"Optimizing for {n_comp}D embeddings...")
        print(f"{'─' * 70}")
        
        embeddings_reduced = umap_cache[n_comp]
        dim_trials = []
        
        def objective(trial):
            min_cluster_size = trial.suggest_int('min_cluster_size', 10, 250)
            min_samples = trial.suggest_int('min_samples', 5, 80)
            method = trial.suggest_categorical('cluster_selection_method', ['eom', 'leaf'])
            
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='euclidean',
                cluster_selection_method=method,
                gen_min_span_tree=True
            )
            labels = clusterer.fit_predict(embeddings_reduced)
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            pct_clustered = (labels != -1).sum() / len(labels)
            
            if n_clusters >= 2 and pct_clustered > 0.1:
                mask = labels != -1
                sil = silhouette_score(embeddings_reduced[mask], labels[mask])
            else:
                sil = -1
            
            dbcv = clusterer.relative_validity_
            
            trial.set_user_attr('n_clusters', n_clusters)
            trial.set_user_attr('pct_clustered', pct_clustered)
            trial.set_user_attr('silhouette', sil)
            trial.set_user_attr('dbcv', dbcv)
            
            dim_trials.append({
                'n_components': n_comp,
                'min_cluster_size': min_cluster_size,
                'min_samples': min_samples,
                'method': method,
                'n_clusters': n_clusters,
                'pct_clustered': pct_clustered,
                'silhouette': sil,
                'dbcv': dbcv
            })
            
            if OPTIMIZE_FOR == 'silhouette':
                return sil if sil > 0 else -1
            elif OPTIMIZE_FOR == 'dbcv':
                return dbcv
            elif OPTIMIZE_FOR == 'balanced':
                if sil <= 0 or n_clusters < 3 or n_clusters > 50 or pct_clustered < 0.3:
                    return -1
                return sil * pct_clustered
            return sil
        
        # Run optimization for this dimensionality
        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=N_TRIALS_PER_DIM, show_progress_bar=True)
        
        # Store results
        all_results.extend(dim_trials)
        best = study.best_trial
        best_per_dim[n_comp] = {
            'n_components': n_comp,
            'min_cluster_size': best.params['min_cluster_size'],
            'min_samples': best.params['min_samples'],
            'method': best.params['cluster_selection_method'],
            'silhouette': best.user_attrs['silhouette'],
            'dbcv': best.user_attrs['dbcv'],
            'n_clusters': best.user_attrs['n_clusters'],
            'pct_clustered': best.user_attrs['pct_clustered']
        }
        
        print(f"  Best: sil={best.user_attrs['silhouette']:.3f}, "
              f"clusters={best.user_attrs['n_clusters']}, "
              f"coverage={best.user_attrs['pct_clustered']*100:.0f}%")
    
    # ============================================================
    # STEP 3: Compare results across dimensionalities
    # ============================================================
    print(f"\n" + "=" * 70)
    print("STEP 3: COMPARISON ACROSS DIMENSIONALITIES")
    print("=" * 70)
    
    comparison_df = pd.DataFrame(best_per_dim.values())
    comparison_df['pct_clustered_str'] = (comparison_df['pct_clustered'] * 100).round(1).astype(str) + '%'
    comparison_df['score'] = comparison_df['silhouette'] * comparison_df['pct_clustered']
    comparison_df = comparison_df.sort_values('silhouette', ascending=False)
    
    print("\nBest result per dimensionality (sorted by silhouette):")
    print(comparison_df[['n_components', 'min_cluster_size', 'min_samples', 'method', 
                         'n_clusters', 'pct_clustered_str', 'silhouette', 'dbcv']].to_string(index=False))
    
    # Find overall best
    best_dim = comparison_df.iloc[0]['n_components']
    overall_best = best_per_dim[best_dim]
    
    print(f"\n" + "=" * 70)
    print(f"OVERALL BEST: {int(best_dim)}D")
    print("=" * 70)
    print(f"  n_components: {int(overall_best['n_components'])}")
    print(f"  min_cluster_size: {overall_best['min_cluster_size']}")
    print(f"  min_samples: {overall_best['min_samples']}")
    print(f"  method: {overall_best['method']}")
    print(f"  silhouette: {overall_best['silhouette']:.4f}")
    print(f"  dbcv: {overall_best['dbcv']:.4f}")
    print(f"  clusters: {overall_best['n_clusters']}")
    print(f"  coverage: {overall_best['pct_clustered']*100:.1f}%")
    
    # Visualization
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Silhouette by Dimensionality', 'Coverage vs Silhouette')
    )
    
    # Box plot of silhouette by dimensionality
    results_df = pd.DataFrame(all_results)
    for n_comp in N_COMPONENTS_OPTIONS:
        comp_sil = results_df[results_df['n_components'] == n_comp]['silhouette']
        fig.add_trace(go.Box(y=comp_sil, name=f'{n_comp}D'), row=1, col=1)
    
    # Scatter: coverage vs silhouette (best per dim)
    fig.add_trace(
        go.Scatter(
            x=[b['pct_clustered']*100 for b in best_per_dim.values()],
            y=[b['silhouette'] for b in best_per_dim.values()],
            mode='markers+text',
            text=[f"{int(b['n_components'])}D" for b in best_per_dim.values()],
            textposition='top center',
            marker=dict(size=12),
            name='Best per dim'
        ),
        row=1, col=2
    )
    
    fig.update_layout(height=400, width=1000, showlegend=False)
    fig.update_xaxes(title_text='UMAP Dimensions', row=1, col=1)
    fig.update_yaxes(title_text='Silhouette Score', row=1, col=1)
    fig.update_xaxes(title_text='Coverage %', row=1, col=2)
    fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
    fig.show()
    
    # Store best params for next cell
    BEST_N_COMPONENTS = int(overall_best['n_components'])
    BEST_MIN_CLUSTER_SIZE = overall_best['min_cluster_size']
    BEST_MIN_SAMPLES = overall_best['min_samples']
    BEST_METHOD = overall_best['method']
    
    print(f"\n✓ Best params saved. Run next cell to apply to FULL dataset.")
    
else:
    print("HDBSCAN not available")

## Apply Optimized Parameters to Full Dataset

The previous cell optimized on a **subsample** (3,000 points) for speed.
Now we apply the best-found parameters to the **full dataset** and compute cluster centroids.

In [ ]:
# Apply optimized UMAP + HDBSCAN to FULL dataset and compute centroids

if HAS_HDBSCAN:
    print("=" * 70)
    print("APPLYING OPTIMIZED UMAP + HDBSCAN TO FULL DATA")
    print("=" * 70)
    
    # Use Optuna-found params if available, otherwise use defaults
    try:
        UMAP_N_COMP = BEST_N_COMPONENTS
        HDBSCAN_MIN_CLUSTER_SIZE = BEST_MIN_CLUSTER_SIZE
        HDBSCAN_MIN_SAMPLES = BEST_MIN_SAMPLES
        HDBSCAN_METHOD = BEST_METHOD
        print("\n  Using Optuna-optimized parameters")
    except NameError:
        # Defaults if Optuna wasn't run
        UMAP_N_COMP = 10
        HDBSCAN_MIN_CLUSTER_SIZE = 100
        HDBSCAN_MIN_SAMPLES = 50
        HDBSCAN_METHOD = 'eom'
        print("\n  Using default parameters (run Optuna cell for optimization)")
    
    print(f"\nParameters:")
    print(f"  UMAP n_components: {UMAP_N_COMP}")
    print(f"  HDBSCAN min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}")
    print(f"  HDBSCAN min_samples: {HDBSCAN_MIN_SAMPLES}")
    print(f"  HDBSCAN method: {HDBSCAN_METHOD}")
    print(f"  Full data shape: {embeddings.shape}")
    
    # Step 1: UMAP reduction on full data
    print(f"\nStep 1: Reducing {embeddings.shape[1]}D → {UMAP_N_COMP}D with UMAP...")
    umap_reducer = umap.UMAP(
        n_neighbors=15,
        n_components=UMAP_N_COMP,
        min_dist=0.0,
        metric='cosine',
        random_state=RANDOM_STATE,
        n_jobs=1
    )
    embeddings_for_hdbscan = umap_reducer.fit_transform(embeddings)
    print(f"  Done: {embeddings_for_hdbscan.shape}")
    
    # Step 2: HDBSCAN clustering
    print(f"\nStep 2: Running HDBSCAN...")
    hdbscan_clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric='euclidean',
        cluster_selection_method=HDBSCAN_METHOD,
        gen_min_span_tree=True,
        prediction_data=True  # Enable approximate_predict for new data
    )
    hdbscan_labels = hdbscan_clusterer.fit_predict(embeddings_for_hdbscan)
    
    # Results
    n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
    n_noise = (hdbscan_labels == -1).sum()
    pct_clustered = (hdbscan_labels != -1).sum() / len(hdbscan_labels) * 100
    
    # Calculate metrics
    mask = hdbscan_labels != -1
    if n_clusters >= 2:
        sil_score = silhouette_score(embeddings_for_hdbscan[mask], hdbscan_labels[mask])
    else:
        sil_score = -1
    dbcv_score = hdbscan_clusterer.relative_validity_
    
    print(f"\n" + "=" * 70)
    print(f"RESULTS (Full Dataset)")
    print("=" * 70)
    print(f"  Clusters found: {n_clusters}")
    print(f"  Noise points: {n_noise:,} ({100-pct_clustered:.1f}%)")
    print(f"  Clustered points: {len(hdbscan_labels) - n_noise:,} ({pct_clustered:.1f}%)")
    print(f"  Silhouette score: {sil_score:.3f}")
    print(f"  DBCV score: {dbcv_score:.3f}")
    
    print(f"\nCluster distribution:")
    for i in sorted(set(hdbscan_labels)):
        count = (hdbscan_labels == i).sum()
        label = "Noise" if i == -1 else f"Cluster {i:2d}"
        pct = count / len(hdbscan_labels) * 100
        bar = "█" * int(pct / 2)
        print(f"  {label}: {count:5,} ({pct:5.1f}%) {bar}")
    
    # Step 3: Compute cluster centroids (in UMAP space)
    print(f"\n" + "=" * 70)
    print("COMPUTING CLUSTER CENTROIDS")
    print("=" * 70)
    
    cluster_centroids = {}
    cluster_metadata_dict = {}
    
    for cluster_id in sorted(set(hdbscan_labels)):
        if cluster_id == -1:
            continue  # Skip noise
        
        cluster_mask = hdbscan_labels == cluster_id
        cluster_embeddings = embeddings_for_hdbscan[cluster_mask]
        
        # Centroid is mean of all points in cluster
        centroid = cluster_embeddings.mean(axis=0)
        cluster_centroids[cluster_id] = centroid
        
        # Collect metadata for this cluster
        cluster_meta = metadata[cluster_mask]
        
        # Get top tools and artifacts
        tool_counts = cluster_meta['tool_name'].value_counts().head(5).to_dict() if 'tool_name' in cluster_meta.columns else {}
        artifact_counts = cluster_meta['artifact_name'].value_counts().head(5).to_dict() if 'artifact_name' in cluster_meta.columns else {}
        
        cluster_metadata_dict[cluster_id] = {
            'size': int(cluster_mask.sum()),
            'top_tools': tool_counts,
            'top_artifacts': artifact_counts
        }
    
    print(f"  Computed centroids for {len(cluster_centroids)} clusters")
    print(f"\n" + "=" * 70)
else:
    hdbscan_labels = None
    embeddings_for_hdbscan = None
    print("HDBSCAN not available")

## Save Cluster Model

Save the trained model for use in `oss_cluster_assignment.ipynb`.

In [ ]:
# Save cluster model for OSS artifact assignment

if HAS_HDBSCAN and hdbscan_labels is not None:
    # Create output directory
    CLUSTER_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    
    # Build cluster model dict (compatible with oss_cluster_assignment.ipynb)
    cluster_model = {
        'created_at': datetime.now().isoformat(),
        'n_training_samples': len(embeddings),
        'n_clusters': len(set(hdbscan_labels) - {-1}),
        'embedding_dim': embeddings.shape[1],
        'parameters': {
            'umap_n_components': UMAP_N_COMP,
            'umap_n_neighbors': 15,
            'umap_min_dist': 0.0,
            'min_cluster_size': HDBSCAN_MIN_CLUSTER_SIZE,
            'min_samples': HDBSCAN_MIN_SAMPLES,
            'cluster_selection_method': HDBSCAN_METHOD,
        },
        'umap_reducer': umap_reducer,           # Fitted UMAP for transform()
        'hdbscan_clusterer': hdbscan_clusterer, # Fitted HDBSCAN for approximate_predict()
        'cluster_centroids': cluster_centroids,  # {cluster_id: centroid_array}
        'cluster_metadata': cluster_metadata_dict,    # {cluster_id: {size, top_tools, top_artifacts}}
    }
    
    # Save with descriptive filename
    mcs = HDBSCAN_MIN_CLUSTER_SIZE
    ms = HDBSCAN_MIN_SAMPLES
    umap_nc = UMAP_N_COMP
    nc = cluster_model['n_clusters']
    
    model_filename = f'cluster_model_mcs{mcs}_ms{ms}_umap{umap_nc}_nc{nc}.pkl'
    model_path = CLUSTER_MODEL_DIR / model_filename
    
    with open(model_path, 'wb') as f:
        pickle.dump(cluster_model, f)
    
    # Also save as "latest"
    with open(CLUSTER_MODEL_DIR / 'cluster_model.pkl', 'wb') as f:
        pickle.dump(cluster_model, f)
    
    print(f"✓ Saved: {model_path}")
    print(f"✓ Saved: {CLUSTER_MODEL_DIR / 'cluster_model.pkl'} (latest)")
    print(f"\nModel contents:")
    print(f"  - n_training_samples: {cluster_model['n_training_samples']:,}")
    print(f"  - n_clusters: {cluster_model['n_clusters']}")
    print(f"  - embedding_dim: {cluster_model['embedding_dim']}")
    print(f"  - UMAP reducer: fitted")
    print(f"  - HDBSCAN clusterer: fitted")
    print(f"  - Cluster centroids: {len(cluster_centroids)} centroids")
else:
    print("No cluster model to save (HDBSCAN not run)")

## UMAP Visualization

In [ ]:
# Reduce to 2D for visualization
# n_jobs=1 prevents OpenMP threading crash on macOS
print("Computing UMAP projection to 2D...")
reducer_2d = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=2,
    metric='cosine',
    random_state=RANDOM_STATE,
    n_jobs=1  # Prevent OMP threading crash
)
umap_2d = reducer_2d.fit_transform(embeddings)

print(f"UMAP complete: {umap_2d.shape}")

In [ ]:
# Build visualization DataFrame
viz_df = pd.DataFrame({
    'global_file_id': global_file_ids,
    'umap_x': umap_2d[:, 0],
    'umap_y': umap_2d[:, 1],
})

if hdbscan_labels is not None:
    viz_df['hdbscan_cluster'] = hdbscan_labels

# Merge with metadata
viz_df = viz_df.merge(
    metadata[['global_file_id', 'artifact_name', 'artifact_path', 'tool_name', 'repo_name']], 
    on='global_file_id', 
    how='left'
)

viz_df.head()

In [ ]:
# Visualization: HDBSCAN clusters
if hdbscan_labels is not None:
    # Create string labels for better legend
    viz_df['hdbscan_label'] = viz_df['hdbscan_cluster'].apply(
        lambda x: 'Noise' if x == -1 else f'Cluster {x}'
    )
    
    # Sort so noise appears last in legend
    cluster_order = ['Noise'] + [f'Cluster {i}' for i in sorted(set(hdbscan_labels)) if i != -1]
    
    n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
    n_noise = (hdbscan_labels == -1).sum()
    pct_noise = n_noise / len(hdbscan_labels) * 100
    
    fig_hdbscan = px.scatter(
        viz_df,
        x='umap_x',
        y='umap_y',
        color='hdbscan_label',
        category_orders={'hdbscan_label': cluster_order},
        hover_data=['artifact_name', 'artifact_path', 'repo_name', 'tool_name'],
        title=f'HDBSCAN Clusters (Optimized)<br><sup>{n_clusters} clusters, {n_noise:,} noise points ({pct_noise:.1f}%)</sup>',
        width=1000,
        height=700,
    )
    
    # Make noise points smaller and gray
    fig_hdbscan.for_each_trace(
        lambda t: t.update(marker=dict(size=4, opacity=0.3)) if t.name == 'Noise' else t.update(marker=dict(size=6, opacity=0.8))
    )
    
    fig_hdbscan.update_layout(
        legend_title='Cluster',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02
        )
    )
    fig_hdbscan.show()
else:
    print("HDBSCAN not available - run the HDBSCAN cell first")

## Cluster Analysis

For each cluster, we'll examine:
1. File names and paths (what patterns emerge?)
2. Tool distribution (which AI tools dominate?)
3. Repository distribution (cluster-specific or cross-repo?)

In [ ]:
def analyze_cluster(df: pd.DataFrame, cluster_id: int, cluster_col: str = 'hdbscan_cluster'):
    """Analyze contents of a single cluster."""
    cluster_df = df[df[cluster_col] == cluster_id]
    
    print(f"\n{'='*60}")
    if cluster_id == -1:
        print(f"NOISE: {len(cluster_df)} files")
    else:
        print(f"CLUSTER {cluster_id}: {len(cluster_df)} files")
    print(f"{'='*60}")
    
    # Tool distribution
    print(f"\nTool distribution:")
    for tool, count in cluster_df['tool_name'].value_counts().head(5).items():
        print(f"  {tool}: {count} ({count/len(cluster_df)*100:.1f}%)")
    
    # Repository distribution
    print(f"\nRepository distribution:")
    for repo, count in cluster_df['repo_name'].value_counts().head(5).items():
        print(f"  {repo}: {count} ({count/len(cluster_df)*100:.1f}%)")
    
    # Common file names
    print(f"\nMost common file names:")
    for name, count in cluster_df['artifact_name'].value_counts().head(10).items():
        print(f"  {name}: {count}")
    
    # Path patterns
    print(f"\nSample paths:")
    for path in cluster_df['artifact_path'].head(10).values:
        print(f"  {path}")
    
    return cluster_df

# Analyze each HDBSCAN cluster (skip noise with -1)
if hdbscan_labels is not None:
    for i in sorted(set(hdbscan_labels)):
        if i != -1:  # Skip noise for detailed analysis
            analyze_cluster(viz_df, i, 'hdbscan_cluster')
else:
    print("Run HDBSCAN clustering first")

## Save Cluster Assignments

Save the cluster assignments with UMAP coordinates for downstream analysis.

In [ ]:
# Save cluster assignments
if hdbscan_labels is not None:
    assignments_df = metadata.copy()
    assignments_df['cluster'] = hdbscan_labels
    assignments_df['umap_x'] = umap_2d[:, 0]
    assignments_df['umap_y'] = umap_2d[:, 1]
    
    assignments_path = DATA_DIR / 'cluster_assignments.csv'
    assignments_df.to_csv(assignments_path, index=False)
    print(f"✓ Saved: {assignments_path}")
    print(f"\nAssignments summary:")
    print(f"  Total artifacts: {len(assignments_df):,}")
    print(f"  Clusters: {len(set(hdbscan_labels) - {-1})}")
    print(f"  Noise points: {(assignments_df['cluster'] == -1).sum():,}")
else:
    print("No assignments to save (HDBSCAN not run)")